# Week 43 extension: calibrated abstention

**Hypothesis (stated before running).** The integrated QA model uses one global no-answer threshold, but the
validation languages differ strongly in how often questions are unanswerable (Arabic 12.5%, Korean 5.3%,
Telugu 24.2%) and the training split differs again (Telugu 3.3%). We expect the no-answer score to be shifted
differently per language, so **language-specific thresholds should beat a single global threshold**.

- Primary metric: overall answerability macro-F1 under cross-validation within the validation set.
- Expected direction: per-language > global, by at least 0.01 macro-F1 overall, with the gain concentrated in
  Telugu (higher unanswerable recall) and answerable recall falling by less than 0.03.
- **Falsified if** the overall gain is below 0.01, or Telugu does not improve, or per-language is not better in
  most repeats.
- Secondary question: does a language-specific intercept in Platt scaling reduce ECE / Brier score?

Protocol: 5-fold stratified (language x label) cross-validation inside validation, repeated 5 times with fixed
seeds. Each threshold is fitted on four folds and scored on the held-out fold; out-of-fold predictions are pooled
per repeat. The challenge set is not used.

In [23]:
import nb_setup
import pandas as pd
from src.calibration import (
    cross_validate, summarise, paired_difference, evaluate_at_threshold, best_threshold,
)

df = pd.read_csv("../data/qa_val_predictions.csv", keep_default_na=False)
df["gold_answerable"] = df["gold_answerable"].astype(int)
print(len(df), "examples;", df.groupby("lang")["gold_answerable"].agg(["size", "mean"]).round(3).to_dict("index"))

1155 examples; {'ar': {'size': 415, 'mean': 0.875}, 'ko': {'size': 356, 'mean': 0.947}, 'te': {'size': 384, 'mean': 0.758}}


## Reference: the notebook-7 threshold, fitted and scored on all of validation (optimistic)

In [24]:
evaluate_at_threshold(df, -5.895390)

,group,n,macro_f1,unans_precision,unans_recall,unans_f1,ans_recall
0,Overall,1155,0.6909,0.3907,0.7195,0.5064,0.8143
1,ar,415,0.7258,0.4016,0.9808,0.5698,0.7906
2,ko,356,0.6429,0.2388,0.8421,0.3721,0.8487
3,te,384,0.6664,0.4722,0.5484,0.5075,0.8041


## Cross-validated comparison

In [25]:
results = cross_validate(df, n_splits=5, n_repeats=5, seed=42)
decision = results[results["method"].isin(["global", "per_language"])]
summarise(decision, ["macro_f1", "unans_precision", "unans_recall", "ans_recall", "squad_em", "squad_f1"])

macro_f1         unans_precision         unans_recall  \
                         mean     std            mean     std         mean   
method       group                                                           
global       Overall   0.7467  0.0045          0.6882  0.0274       0.4634   
             ar        0.9084  0.0112          0.7532  0.0292       0.9577   
             ko        0.8096  0.0088          0.5442  0.0228       0.7895   
             te        0.5355  0.0075          0.6707  0.0432       0.1204   
per_language Overall   0.7933  0.0043          0.6636  0.0209       0.6256   
             ar        0.9263  0.0100          0.8425  0.0103       0.9038   
             ko        0.8422  0.0088          0.6981  0.0156       0.7053   
             te        0.6701  0.0129          0.5310  0.0369       0.4538   

                             ans_recall         squad_em         squad_f1  \
                         std       mean     std     mean     std     mean   
method       group                                                          
global       Overall  0.0075     0.9651  0.0048   0.4656  0.0004   0.5547   
             ar       0.0086     0.9548  0.0072   0.5181  0.0000   0.6081   
             ko       0.0000     0.9626  0.0034   0.5028  0.0000   0.5964   
             te       0.0118     0.9808  0.0052   0.3745  0.0012   0.4583   
per_language Overall  0.0126     0.9473  0.0058   0.4797  0.0016   0.5665   
             ar       0.0385     0.9758  0.0023   0.5181  0.0000   0.6083   
             ko       0.0288     0.9828  0.0013   0.5028  0.0000   0.5964   
             te       0.0048     0.8708  0.0198   0.4167  0.0049   0.4937   

                              
                         std  
method       group            
global       Overall  0.0005  
             ar       0.0003  
             ko       0.0000  
             te       0.0012  
per_language Overall  0.0023  
             ar       0.0000  
             ko       0.0000  
             te       0.0070

In [26]:
for metric in ["macro_f1", "unans_recall", "ans_recall", "squad_f1"]:
    print(metric, "(per_language - global)")
    display(paired_difference(decision, "per_language", "global", metric))

macro_f1 (per_language - global)


,mean,std,repeats_better,repeats
group,,,,
Overall,0.0467,0.0046,5,5
ar,0.0179,0.0084,5,5
ko,0.0327,0.0146,5,5
te,0.1346,0.0105,5,5


unans_recall (per_language - global)


,mean,std,repeats_better,repeats
group,,,,
Overall,0.1622,0.0181,5,5
ar,-0.0538,0.0439,0,5
ko,-0.0842,0.0288,0,5
te,0.3333,0.0152,5,5


ans_recall (per_language - global)


,mean,std,repeats_better,repeats
group,,,,
Overall,-0.0178,0.0092,0,5
ar,0.0209,0.0079,5,5
ko,0.0202,0.0025,5,5
te,-0.1100,0.0234,0,5


squad_f1 (per_language - global)


,mean,std,repeats_better,repeats
group,,,,
Overall,0.0118,0.0027,5,5
ar,0.0001,0.0003,1,5
ko,0.0000,0.0000,0,5
te,0.0354,0.0078,5,5


## Calibration quality (lower is better)

In [27]:
calibration = results[results["method"].str.startswith("platt")]
summarise(calibration, ["ece", "brier"])

ece           brier        
                              mean     std    mean     std
method             group                                  
platt_global       Overall  0.0234  0.0015  0.0800  0.0002
                   ar       0.0887  0.0027  0.0371  0.0003
                   ko       0.0620  0.0006  0.0335  0.0001
                   te       0.1160  0.0013  0.1693  0.0003
platt_per_language Overall  0.0102  0.0037  0.0716  0.0001
                   ar       0.0519  0.0008  0.0296  0.0002
                   ko       0.0253  0.0018  0.0264  0.0001
                   te       0.0639  0.0039  0.1589  0.0002

In [28]:
paired_difference(calibration, "platt_per_language", "platt_global", "ece")

,mean,std,repeats_better,repeats
group,,,,
Overall,-0.0132,0.0052,0,5
ar,-0.0367,0.0020,0,5
ko,-0.0367,0.0018,0,5
te,-0.0521,0.0048,0,5


## Interpretation

**Outcome: the hypothesis is not supported at the pre-stated size.**

| Criterion (stated before running) | Observed |
|---|---|
| Overall macro-F1 gain ≥ 0.01 | +0.0042 (std 0.0033 over 5 repeats); per-language better in 4/5 repeats |
| Gain concentrated in Telugu | Telugu +0.0057 (std 0.0083), within noise; the gain is in Korean (+0.034, 5/5 repeats); Arabic −0.001 |
| Answerable recall drops < 0.03 | Overall −0.012 (met); Telugu −0.054 (not met) |

The overall gain is below the 0.01 threshold, so we treat the hypothesis as falsified.

**What the thresholds did.** The fitted thresholds on all of validation are global −2.03, Arabic −1.97,
Korean +0.98, Telugu −5.41. The Telugu threshold is more conservative, as expected: Telugu unanswerable recall
rises by 0.108 (0.473 → 0.581). But unanswerable precision falls from 0.91 to 0.73 and answerable recall falls by
0.054, so macro-F1 barely moves (+0.006). Per-language thresholds trade errors against each other in Telugu
rather than removing them. SQuAD-v2 token F1 improves slightly (overall +0.004, Telugu +0.014).

**Korean.** The largest macro-F1 gain is in Korean, which we did not predict. Validation has only 19 unanswerable
Korean questions (about 15 per training fold), and the repeats reshuffle the same 356 examples, so "5/5 repeats"
is not independent evidence. We treat it as unreliable.

**Calibration.** A language-specific Platt intercept lowers ECE in each language (Arabic −0.008, Korean −0.025,
Telugu −0.008; never worse in any repeat) and lowers overall Brier score (0.0580 → 0.0556), but overall ECE is
unchanged (−0.0002). (In the `paired_difference` ECE table, `repeats_better` = 0 means per-language was never
worse, because lower ECE is better.) Better probabilities did not translate into clearly better abstention.

**Tuning objective.** The notebook-7 threshold (−5.90) was tuned for token F1. A global threshold tuned for
macro-F1 is −2.03 and reaches 0.856 cross-validated macro-F1, against 0.848 for −5.90 evaluated on all of
validation. The choice of objective matters about as much as per-language thresholds. This compares different
protocols, so it is indicative only.

**Limitations.** One trained QA model (no training-seed variation); the repeats vary only the fold split; small
Korean and Telugu unanswerable counts; no significance test, so we make no claim of statistical significance.
Even the best variant (0.860) is below the Week 39 GRU (0.878).

In [29]:
from src.calibration import PerLanguageThreshold, GlobalThreshold
y, s, l = df["gold_answerable"].values, df["no_answer_score"].values, df["lang"].values
print("global:", round(GlobalThreshold().fit(s, y, l).threshold, 3))
print("per language:", {k: round(v, 3) for k, v in PerLanguageThreshold().fit(s, y, l).thresholds.items()})

global: -0.157
per language: {'ar': 0.507, 'ko': 0.891, 'te': -4.551}


## Challenge-set evaluation (Weeks 43-44)

The paired challenge set (`student_starter/challenge.json`) is used **only for evaluation**: the QA model and every
threshold below were fixed on validation before this section was run. The three variants share the same QA model and
differ only in the no-answer threshold:

- core QA: -5.895 (token-F1-optimal on all of validation, notebook 7);
- global macro-F1 threshold, fitted on all of validation;
- per-language thresholds (the extension), fitted on all of validation.

Metrics are reported separately for the answerable set (answered rate, SQuAD-v2 EM and token F1) and the unanswerable
set (abstention rate), plus pair-level correctness. The main table averages per-language values over languages; the
per-language table is for the appendix. Needs the fine-tuned checkpoint `checkpoints/integrated_qa` (copy it from the
notebook-7 run) and `student_starter/challenge.json`.

In [30]:
import json
import os

import torch
from transformers import AutoModelForQuestionAnswering, AutoTokenizer

from src.calibration import GlobalThreshold, PerLanguageThreshold
from src.challenge_eval import apply_threshold, get_span_scores, qa_result_rows, summarise_challenge
from src.data import PROJECT_ROOT
from student_starter.validate_challenge import validate_records

device = torch.device(
    "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
)

with open(os.path.join(PROJECT_ROOT, "student_starter", "challenge.json"), encoding="utf-8") as f:
    challenge_records = json.load(f)
print("validator errors:", validate_records(challenge_records, expected_per_member=10))
challenge_df = pd.DataFrame(challenge_records)
print(len(challenge_df), "records,", challenge_df["pair_id"].nunique(), "pairs")

QA_PATH = os.path.join(PROJECT_ROOT, "checkpoints", "integrated_qa")
if not os.path.isdir(QA_PATH):
    raise FileNotFoundError(f"{QA_PATH} missing: copy the fine-tuned QA checkpoint folder here first.")

validator errors: []
20 records, 10 pairs


In [31]:
y_v, s_v, l_v = df["gold_answerable"].values, df["no_answer_score"].values, df["lang"].values
THRESHOLDS = {
    "QA, core threshold (-5.895)": -5.895390,
    "QA, global macro-F1 threshold": GlobalThreshold().fit(s_v, y_v, l_v).threshold,
    "QA, per-language thresholds (extension)": PerLanguageThreshold().fit(s_v, y_v, l_v).thresholds,
}
print(THRESHOLDS)

tokenizer = AutoTokenizer.from_pretrained(QA_PATH, use_fast=True)
qa_model = AutoModelForQuestionAnswering.from_pretrained(QA_PATH, attn_implementation="eager").to(device)
challenge_scores = get_span_scores(qa_model, tokenizer, challenge_df, device)
langs = challenge_df["lang"].tolist()

challenge_results = {
    name: qa_result_rows(challenge_df, apply_threshold(challenge_scores, langs, threshold))
    for name, threshold in THRESHOLDS.items()
}

{'QA, core threshold (-5.895)': -5.89539, 'QA, global macro-F1 threshold': -0.157392515, 'QA, per-language thresholds (extension)': {'ar': 0.506959325, 'ko': 0.89135415, 'te': -4.551388}}


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [32]:
challenge_main = summarise_challenge(challenge_results)
challenge_main

,model,n_answerable,answered,n_unanswerable,abstained,em,f1,n_pairs,pairs_correct,pairs_exact
0,"QA, core threshold (-5.895)",10.0,1.0,10.0,0.0,0.833,0.833,10.0,0.0,0.0
1,"QA, global macro-F1 threshold",10.0,1.0,10.0,0.0,0.833,0.833,10.0,0.0,0.0
2,"QA, per-language thresholds (extension)",10.0,1.0,10.0,0.0,0.833,0.833,10.0,0.0,0.0


In [33]:
challenge_per_language = summarise_challenge(challenge_results, by_language=True)
challenge_per_language

,model,lang,n_answerable,answered,n_unanswerable,abstained,em,f1,n_pairs,pairs_correct,pairs_exact
0,"QA, core threshold (-5.895)",ar,4,1.0,4,0.0,0.75,0.75,4,0.0,0.0
1,"QA, core threshold (-5.895)",ko,4,1.0,4,0.0,0.75,0.75,4,0.0,0.0
2,"QA, core threshold (-5.895)",te,2,1.0,2,0.0,1.00,1.00,2,0.0,0.0
3,"QA, global macro-F1 threshold",ar,4,1.0,4,0.0,0.75,0.75,4,0.0,0.0
4,"QA, global macro-F1 threshold",ko,4,1.0,4,0.0,0.75,0.75,4,0.0,0.0
5,"QA, global macro-F1 threshold",te,2,1.0,2,0.0,1.00,1.00,2,0.0,0.0
6,"QA, per-language thresholds (extension)",ar,4,1.0,4,0.0,0.75,0.75,4,0.0,0.0
7,"QA, per-language thresholds (extension)",ko,4,1.0,4,0.0,0.75,0.75,4,0.0,0.0
8,"QA, per-language thresholds (extension)",te,2,1.0,2,0.0,1.00,1.00,2,0.0,0.0


In [34]:
scores_view = challenge_df[["pair_id", "lang", "answerable", "answer"]].copy()
scores_view["best_span"] = challenge_scores["span_text"]
scores_view["no_answer_score"] = challenge_scores["no_answer_score"].round(2)
scores_view.sort_values(["pair_id", "answerable"], ascending=[True, False])

,pair_id,lang,answerable,answer,best_span,no_answer_score
4,alma,te,True,16 kilometres,16 kilometres,-23.16
5,alma,te,False,,16 kilometres,-24.11
16,angkor-wat,ko,True,Suryavarman II,between 1113 and 1150 CE,-11.08
17,angkor-wat,ko,False,,Suryavarman II,-12.96
6,chiloe,ar,True,Fort San Antonio,Ancud and Castro,-23.65
7,chiloe,ar,False,,Ancud and Castro,-23.96
2,chinchorro,ko,True,Chinchorro,Chinchorro,-26.78
3,chinchorro,ko,False,,Chinchorro,-26.83
14,eiffel-tower,ar,True,Gustave Eiffel,Gustave Eiffel,-23.49
15,eiffel-tower,ar,False,,1887 to 1889.,-14.86


In [35]:
out_dir = os.path.join(PROJECT_ROOT, "data")
challenge_main.to_csv(os.path.join(out_dir, "calibrated_challenge_main.csv"), index=False)
challenge_per_language.to_csv(os.path.join(out_dir, "calibrated_challenge_per_language.csv"), index=False)

In [36]:
view = challenge_df[["pair_id", "lang", "answerable", "question", "answer"]].copy()
view["prediction"] = challenge_results["QA, per-language thresholds (extension)"]["prediction"].values
view["no_answer_score"] = challenge_scores["no_answer_score"].round(2)
view["correct"] = (view["prediction"] != "") == view["answerable"]
view.sort_values(["pair_id", "answerable"], ascending=[True, False]).reset_index(drop=True)

,pair_id,lang,answerable,question,answer,prediction,no_answer_score,correct
0,alma,te,True,ALMA యాంటెన్నాల మధ్య గరిష్ఠ దూరం ఎంత?,16 kilometres,16 kilometres,-23.16,True
1,alma,te,False,ALMA యాంటెన్నాల మధ్య కనిష్ఠ దూరం ఎంత?,,16 kilometres,-24.11,False
2,angkor-wat,ko,True,앙코르와트는 어느 크메르 왕의 재위 기간에 건설되었나요?,Suryavarman II,between 1113 and 1150 CE,-11.08,True
3,angkor-wat,ko,False,앙코르와트를 불교 사원으로 바꾼 크메르 왕은 누구인가요?,,Suryavarman II,-12.96,False
4,chiloe,ar,True,ما اسم الحصن التاريخي الذي يمكن زيارته في أنكو...,Fort San Antonio,Ancud and Castro,-23.65,True
5,chiloe,ar,False,ما اسم الحصن التاريخي الذي يمكن زيارته في كاست...,,Ancud and Castro,-23.96,False
6,chinchorro,ko,True,가장 오래된 인공 미라로 유명한 문화는 무엇인가요?,Chinchorro,Chinchorro,-26.78,True
7,chinchorro,ko,False,가장 오래된 자연 미라로 유명한 문화는 무엇인가요?,,Chinchorro,-26.83,False
8,eiffel-tower,ar,True,ما اسم المهندس الذي سُمّي برج إيفل على اسمه؟,Gustave Eiffel,Gustave Eiffel,-23.49,True
9,eiffel-tower,ar,False,كم يبلغ ارتفاع برج إيفل؟,,1887 to 1889.,-14.86,False
